# 07. Custom Tensor Subclasses Quantized and Tracked Tensors

- **Category:** 17_deployment_low_level_and_capstones
- **Lineage & Citations:** PyTorch Core Team (2022, *Tensor Wrapper Subclasses*), Ansel et al. (2024, *PyTorch 2*).
- **Purpose:** Construct first-class custom Tensor subclasses via `_make_wrapper_subclass` implementing DualTensor for forward-AD Jacobian-vector products and LazyDequantTensor for transparent on-the-fly dequantization.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Torch Dispatch Modes FLOP Counter and Memory Tracker](06_torch_dispatch_modes_flop_counter_and_memory_tracker.ipynb) $\to$ `🔗 Downstream Syntheses:` [Triton and Tiled Kernel Algorithms Softmax RMSNorm Liger Ce](08_triton_and_tiled_kernel_algorithms_softmax_rmsnorm_liger_ce.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Never subclass `torch.Tensor` using standard Python `__init__` inheritance without `_make_wrapper_subclass`. Doing so destroys underlying tensor storage aliasing, breaking autograd view metadata and in-place tensor mutations.

## Mathematical Foundations & Governing Equations

$$
(A + \epsilon \dot{A})(B + \epsilon \dot{B}) = AB + \epsilon(A\dot{B} + \dot{A}B), \quad W_{\text{fp}} = (W_{\text{int}} - z) \cdot s
$$
$$
Y_{\text{dual}} = (X_p W_{\text{fp}} + b) + \epsilon (X_t W_{\text{fp}})
$$

### Asymptotic Complexity & Storage Savings
- **Forward-AD JVP:** Computes exact directional derivatives in $O(1)$ memory without unrolling backward computation graphs.
- **Weight Compression:** INT8 tensor storage cuts memory consumption by $75\%$ compared to standard FP32 weights.

In [ ]:
import torch
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/17_deployment_low_level_and_capstones/07_custom_tensor_subclasses_quantized_and_tracked_tensors.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

class DualTensor(torch.Tensor):
    primal: torch.Tensor
    tangent: torch.Tensor

    @staticmethod
    def __new__(cls, primal, tangent):
        res = torch.Tensor._make_wrapper_subclass(
            cls, primal.shape, strides=primal.stride(), storage_offset=primal.storage_offset(),
            dtype=primal.dtype, layout=primal.layout, device=primal.device, requires_grad=False
        )
        res.primal = primal
        res.tangent = tangent
        return res

    @classmethod
    def __torch_dispatch__(cls, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}
        primal_args = [a.primal if isinstance(a, DualTensor) else a for a in args]
        out_primal = func(*primal_args, **kwargs)
        name = str(func)
        if "mm" in name or "matmul" in name:
            a, b = args[0], args[1]
            a_p = a.primal if isinstance(a, DualTensor) else a
            a_t = a.tangent if isinstance(a, DualTensor) else torch.zeros_like(a_p)
            b_p = b.primal if isinstance(b, DualTensor) else b
            b_t = b.tangent if isinstance(b, DualTensor) else torch.zeros_like(b_p)
            out_tangent = (a_p @ b_t) + (a_t @ b_p)
        elif "add" in name:
            a, b = args[0], args[1]
            a_t = a.tangent if isinstance(a, DualTensor) else torch.zeros_like(a)
            b_t = b.tangent if isinstance(b, DualTensor) else torch.zeros_like(b)
            out_tangent = a_t + b_t
        else:
            out_tangent = torch.zeros_like(out_primal)
        return DualTensor(out_primal, out_tangent)

def dequantize_affine(int_w, scale, zero_point):
    return (int_w.to(torch.float32) - zero_point) * scale

def forward_dual_affine(x_primal, x_tangent, w, b):
    x_dual = DualTensor(x_primal, x_tangent)
    y_dual = x_dual @ w + b
    return y_dual.primal, y_dual.tangent

In [ ]:
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

dequant_w = dequantize_affine(weights["int_w"], weights["scale"], weights["zero_point"])
assert float(torch.max(torch.abs(dequant_w - weights["w"]))) < 1e-5

y_primal, y_tangent = forward_dual_affine(weights["x_primal"], weights["x_tangent"], dequant_w, weights["b"])
assert float(torch.max(torch.abs(y_primal - weights["y_primal"]))) < 1e-5
assert float(torch.max(torch.abs(y_tangent - weights["y_tangent"]))) < 1e-5
print("All Custom Tensor Subclass parity checks passed (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

benchmarks = [
    benchmark_fn(lambda: linear_anchor(weights["test_input"], weights["weight"], weights["bias"]), iters=50, name="Linear Anchor"),
    benchmark_fn(lambda: forward_dual_affine(weights["x_primal"], weights["x_tangent"], weights["w"], weights["b"]), iters=50, name="Dual Forward Affine"),
]
print(format_benchmark_table(benchmarks))

# SVG Visualization of DualTensor Forward AD
svg = """<svg width="480" height="80" xmlns="http://www.w3.org/2000/svg">
  <rect width="480" height="80" fill="#1e1e2e" rx="8"/>
  <rect x="10" y="20" width="120" height="40" fill="#313244" rx="4" stroke="#89b4fa" stroke-width="1.5"/>
  <text x="70" y="44" fill="#cdd6f4" font-size="11" text-anchor="middle">DualTensor (x, x_dot)</text>
  <path d="M135 40 L185 40" stroke="#89b4fa" stroke-width="2"/>
  <rect x="190" y="20" width="120" height="40" fill="#313244" rx="4" stroke="#a6e3a1" stroke-width="1.5"/>
  <text x="250" y="44" fill="#cdd6f4" font-size="11" text-anchor="middle">LazyDequant W</text>
  <path d="M315 40 L355 40" stroke="#a6e3a1" stroke-width="2"/>
  <rect x="360" y="20" width="110" height="40" fill="#313244" rx="4" stroke="#cba6f7" stroke-width="1.5"/>
  <text x="415" y="44" fill="#cdd6f4" font-size="11" text-anchor="middle">Primal + Tangent</text>
</svg>"""
from IPython.display import display, HTML
display(HTML(svg))

In [ ]:
# Pathological Misuse: Naive Eager Materialization vs Lazy Subclass Dequantization
int_weights = torch.randint(-128, 127, (1024, 1024), dtype=torch.int8)
int8_bytes = int_weights.numel() * int_weights.element_size()  # 1 MB

# Naive eager dequantization immediately expands weights to 4 MB
eager_fp32 = int_weights.to(torch.float32) * 0.01
fp32_bytes = eager_fp32.numel() * eager_fp32.element_size()  # 4 MB

print(f"Quantized Subclass Memory:     {int8_bytes / 1024:.1f} KB")
print(f"Eager Materialization Memory: {fp32_bytes / 1024:.1f} KB (4x memory inflation!)")
assert fp32_bytes == 4 * int8_bytes, "Pathology demonstrated!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **First-Class Subclasses:** `_make_wrapper_subclass` allows building specialized tensors that integrate natively into existing PyTorch modules.
2. **Dual Number Forward AD:** DualTensor evaluates directional derivatives without accumulating tape history or backward pass memory.
3. **Transparent Compression:** Lazy dequantization retains INT8 storage density in memory while exposing float32 mathematical semantics.

---
### Cross-Domain Generalisation Matrix
| Domain | Tensor Subclass Analogue | Architectural Objective |
| :--- | :--- | :--- |
| **Robotics Kinematics** | Dual Quaternions (H + eps*H) | Unified rigid body transformation without gimbal lock singularities |
| **Physics Simulations** | Dimensional Analysis (Unit-Checked Tensors) | Compile-time enforcement of physical dimension invariants |
| **Distributed Meshes** | DTensor Placements (Replicated / Sharded) | Seamless multi-node pipeline orchestration without manual collective calls |

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [07_custom_tensor_subclasses_quantized_and_tracked_tensors.md](../../markdown/17_deployment_low_level_and_capstones/07_custom_tensor_subclasses_quantized_and_tracked_tensors.md)
- **PyTorch 2.x Module:** [07_custom_tensor_subclasses_quantized_and_tracked_tensors.py](../../pytorch2/17_deployment_low_level_and_capstones/07_custom_tensor_subclasses_quantized_and_tracked_tensors.py)
- **Functional JAX Module:** [07_custom_tensor_subclasses_quantized_and_tracked_tensors.py](../../jax/17_deployment_low_level_and_capstones/07_custom_tensor_subclasses_quantized_and_tracked_tensors.py)
- **Native Rust Candle Track:** [07_custom_tensor_subclasses_quantized_and_tracked_tensors.rs](../../candle/17_deployment_low_level_and_capstones/07_custom_tensor_subclasses_quantized_and_tracked_tensors.rs)
- **Hardware Kernel (CPU / WGSL):** [07_custom_tensor_subclasses_quantized_and_tracked_tensors.rs](../../kernels/17_deployment_low_level_and_capstones/07_custom_tensor_subclasses_quantized_and_tracked_tensors.rs)
- **Golden SafeTensors:** [07_custom_tensor_subclasses_quantized_and_tracked_tensors.safetensors](../../weights/17_deployment_low_level_and_capstones/07_custom_tensor_subclasses_quantized_and_tracked_tensors.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 06. Torch Dispatch Modes FLOP Counter and Memory Tracker](06_torch_dispatch_modes_flop_counter_and_memory_tracker.ipynb) | [Master Index](../../README.md#17_deployment_low_level_and_capstones) | [Next: 08. Triton and Tiled Kernel Algorithms Softmax RMSNorm Liger Ce](08_triton_and_tiled_kernel_algorithms_softmax_rmsnorm_liger_ce.ipynb) &rarr;